In [1]:
# load graph
import numpy as np
import torch

d = np.load("../data/constructs/canwell_graph.npz", allow_pickle=True)
rows, cols = d["rows"], d["cols"]
elev       = d["elev"]
slope      = d["slope"]
aspect     = d["aspect"]
doubslope  = d["doubslope"]
ddem       = d["ddem"]
is_basin   = d["is_basin"]
is_slope   = d["is_slope"]
edge_index = torch.from_numpy(d["edge_index"]).long()
H, W       = d["shape"]
N = elev.shape[0]


# node-center coordinates in meters (affine: b = d = 0 for a north-up grid)
a, b, c, dd, e, f = d["transform"]
x_coord = a*cols + c + a/2.0       # easting of each node center
y_coord = e*rows + f + e/2.0       # northing of each node center

# label set: slope nodes with a valid target (ddem)
label = is_slope & ~np.isnan(ddem)
print("N:", int(N), "| lableable slope nodes:", int(label.sum()),
      "| basin count:", int(is_basin.sum()))

N: 714421 | lableable slope nodes: 386547 | basin count: 327106


In [2]:
# random split prep
SEED = 42
VAL_FRAC = 0.25
TEST_FRAC = 0.25 # aggressive -- train pool = 0.50
rng = np.random.default_rng(SEED)

lab_idx = np.where(label)[0]         # labelable slope-node indices
perm = rng.permutation(lab_idx.size)
n_test = int(TEST_FRAC * lab_idx.size)
n_val  = int(VAL_FRAC  * lab_idx.size)
test_idx   = lab_idx[perm[:n_test]]
val_idx    = lab_idx[perm[n_test:n_test+n_val]]
train_pool = lab_idx[perm[n_test+n_val:]]      # remaining 50%

In [4]:
# learning curve knob: fraction of training pool actually used (1.0 = all)
# where "all" is the full 50%
TRAIN_FRAC = 1.0
train_idx = rng.permutation(train_pool)[:int(TRAIN_FRAC * train_pool.size)]

print(f"train {train_idx.size:,} | val {val_idx.size:,} | test {test_idx.size:,} "
      f"| train_frac {TRAIN_FRAC}")

train 193,275 | val 96,636 | test 96,636 | train_frac 1.0


In [7]:
# save labeled node splits
np.savez(f"../data/constructs/split_random_s{SEED}.npz",
         train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
         train_pool=train_pool, seed=SEED,
         val_frac=VAL_FRAC, test_frac=TEST_FRAC, train_frac=TRAIN_FRAC)
print("saved")

saved


In [9]:
# static features (fill basin terrain, scale on train stats)
asp = np.deg2rad(aspect)
asp_sin = np.sin(asp).astype(np.float32)
asp_cos = np.cos(asp).astype(np.float32)

def fill_train_mean(arr):
    m = np.nanmean(arr[train_idx])    # train = slope nodes, all real
    out = arr.astype(np.float32).copy() # don't manipulate original array
    out[np.isnan(out)] = np.float32(m)  # basin + any stray NaN -> neutral
                                     # assumes features except elev are NaN in basin
    return out

slope_f = fill_train_mean(slope)
doub_f  = fill_train_mean(doubslope)
sin_f   = fill_train_mean(asp_sin)
cos_f   = fill_train_mean(asp_cos)
elev_f  = elev.astype(np.float32)      # real everywhere, no fill

print("basin and nans filled")

basin and nans filled


In [10]:
stats = {}
def zscore(arr, name):
    m, s = float(arr[train_idx].mean()), float(arr[train_idx].std() + 1e-8)
    stats[name] = (m,s)
    return ((arr - m) / s).astype(np.float32)

elev_z  = zscore(elev_f,  "elev")
slope_z = zscore(slope_f, "slope")
doub_z  = zscore(doub_f,  "doubslope")

X_static = np.column_stack([elev_z, slope_z, sin_f, cos_f, doub_z,
                            is_basin.astype(np.float32)]).astype(np.float32)
print("X_static:", X_static.shape,
      "| basin slope_z mean ~0:", round(float(slope_z[is_basin].mean()), 3),
      "| train slope_z std ~1:", round(float(slope_z[train_idx].std()), 3))

X_static: (714421, 6) | basin slope_z mean ~0: 0.0 | train slope_z std ~1: 1.0


In [12]:
# --- target prep ----
import torch
from torch_geometric.data import Data

# target y: slope DEM, normalized on train stats (0 where no target; unused there)
y_mean = float(np.nanmean(ddem[train_idx]))
y_std  = float(np.nanstd(ddem[train_idx]) + 1e-8)
y = np.zeros(N, np.float32)
vt = ~np.isnan(ddem)
y[vt] = (ddem[vt] - y_mean) / y_std
stats["y"] = (y_mean, y_std)


In [14]:
# diff_input: basin dDEM only (as a feature) -- 0 on all slope nodes (their dDEM is target)
VISIBLE = True                           # False -> no-basin ablation (zeros everywhere)
diff = np.zeros(N, np.float32)
b_mean = b_std = None
if VISIBLE:
    bsig = is_basin & ~np.isnan(ddem)
    b_mean = float(ddem[bsig].mean())
    b_std  = float(ddem[bsig].std() + 1e-8)
    diff[bsig] = (ddem[bsig] - b_mean) / b_std # basin signal has own normalization

stats["diff"] = (b_mean, b_std)

X = np.column_stack([X_static, diff]).astype(np.float32)   # 7 columns

In [16]:
# build data
data = Data(x = torch.from_numpy(X), edge_index=edge_index, y=torch.from_numpy(y))

def bmask(idx):
    m = torch.zeros(N, dtype=torch.bool); m[idx]=True; return m
data.train_mask = bmask(train_idx)
data.val_mask   = bmask(val_idx)
data.test_mask  = bmask(test_idx)

print("x:", tuple(data.x.shape), "| VISIBLE:", VISIBLE,
      "| y mean/std:", round(y_mean, 3), round(y_std, 3),
      "| basin diff mean/std:", None if b_mean is None else (round(b_mean, 3), round(b_std, 3)))
print("masks train/val/test:",
      int(data.train_mask.sum()), int(data.val_mask.sum()), int(data.test_mask.sum()))

x: (714421, 7) | VISIBLE: True | y mean/std: -1.957 3.742 | basin diff mean/std: (-24.991, 15.102)
masks train/val/test: 193275 96636 96636


In [17]:
# 5 — checkpoint prepared data + stats
import json, datetime

np.savez("../data/constructs/staged_s42.npz",
    X_static=X_static, diff=diff, y=y,
    train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
    cols=np.array(["elev","slope","aspect_sin","aspect_cos",
                   "doubslope","is_basin","diff_input"]),
    seed=np.int64(SEED))

staged42_meta = {
    "created": datetime.date.today().isoformat(),
    "graph": "canwell_graph.npz",
    "split": {"seed": SEED, "train_frac": TRAIN_FRAC,
              "n_train": int(train_idx.size), "n_val": int(val_idx.size),
              "n_test": int(test_idx.size)},
    "columns": ["elev","slope","aspect_sin","aspect_cos",
                "doubslope","is_basin","diff_input"],
    "scaling": {k: [round(v[0],6), round(v[1],6)] for k,v in stats.items()},
    "notes": {
        "diff_input": "basin dDEM, basin-normalised; 0 on slope; zeroed for no-basin run",
        "y": "slope dDEM, train-normalised; denorm = y*std + mean",
        "basin_terrain": "neutral-filled with train mean before scaling",
    },
}
with open("../data/constructs/staged_s42.meta.json","w") as fh:
    json.dump(staged42_meta, fh, indent=2)
print("checkpoint written:", X_static.shape, "| stats keys:", list(stats.keys()))

checkpoint written: (714421, 6) | stats keys: ['elev', 'slope', 'doubslope', 'y', 'diff']
